# Chapter 2 - Lab 4: Fetching Stock Price Information Using ReAct Paradigm

In this hands-on lab, we implement an agent using the **ReAct** paradigm.

The agent retrieves the most recent price of a given stock along with the corresponding date by invoking a dedicated tool, get_latest_price, which calls the Yahoo Finance API.

This example relies on the ReAct agent implementation provided by LlamaIndex and uses the gpt-4o model from OpenAI

# Install Packages

In [ ]:
!pip install -U llama-index -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, 

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# ReAct Agent built-in class:

In [ ]:
from llama_index.core.agent.workflow import ReActAgent

## Define the tool

In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.llms.openai import OpenAI


import yfinance as yf

def get_latest_price(ticker_symbol: str):
    """
    Fetch the latest price for the given ticker symbol.
    """
    ticker = yf.Ticker(ticker_symbol)
    data = ticker.history(period="1d", interval="1m")

    if not data.empty:
        latest_row = data.iloc[-1]
        return {
            "ticker": ticker_symbol,
            "datetime": latest_row.name.strftime("%Y-%m-%d %H:%M"),
            "price": latest_row["Close"]
        }
    else:
        return {"error": f"No data found for ticker: {ticker_symbol}"}


tools = [
    FunctionTool.from_defaults(get_latest_price),
]

The `get_latest_price` tool is built on the Yahoo Finance API. It retrieves the most recent historical price for a given ticker along with the corresponding date.

In [ ]:
## Define the tool + the Agent

## Specify the Agent

The `get_latest_price` tool is integrated into the built-in ReAct agent implementation of LlamaIndex.


We specify also the LLM that will be used: GPT-4o

In [ ]:
agent_prices = ReActAgent(
    llm=OpenAI(model="gpt-4o", api_key = OPENAI_API_KEY), tools=tools, timeout=120, verbose=False
)

## Run the Agent

In [ ]:
ret_1 = await agent_prices.run("What is the last price of Apple?")
print(ret_1.response.blocks[0].text)

The latest price of Apple (AAPL) is $234.06 as of September 12, 2025.


We get the last price of Apple at the time of the run of this notebook.

Here is the full object returned by LlamaIndex ReAct agent:

In [ ]:
ret_1

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='The latest price of Apple (AAPL) is $234.06 as of September 12, 2025.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-CFR9PRCe3q2g5TpAP9aNRzaCMa2VH', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1757795323, 'model': 'gpt-4o-2024-08-06', 'object': 'chat.completion.chunk', 'service_tier': 'default', 'system_fingerprint': 'fp_cbf1785567', 'usage': None, 'obfuscation': 'KOcw8Kq'}, tool_calls=[ToolCallResult(tool_name='get_latest_price', tool_kwargs={'ticker_symbol': 'AAPL'}, tool_id='9c346c15-d204-4e4c-b231-6270ab673f3f', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text="{'ticker': 'AAPL', 'datetime': '2025-09-12 15:59', 'price': np.float64(234.05999755859375)}")], too

In [ ]:
print(ret_1.response.blocks[0].text)

The latest price of Apple (AAPL) is $234.06 as of September 12, 2025.


## With more details in the ouput to see the Reasoning + Acting traces:

This snippet streams the agent’s output in real time, displaying incremental text as it is generated, and then awaits the final consolidated response once execution is complete:

In [ ]:
# Import the AgentStream event type used to stream agent responses
from llama_index.core.agent.workflow import AgentStream

# Run the ReAct agent with a user query and obtain a handler for streaming and final response
handler = agent_prices.run("What is the last price of Apple?")

 # Iterate over streamed events produced during agent execution
async for ev in handler.stream_events():
    # Filter for text-generation events from the agent
    if isinstance(ev, AgentStream):
        # Print incremental output as it is generated
        print(ev.delta, end="", flush=True)

# Await the final consolidated response from the agent
response = await handler

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_latest_price
Action Input: {"ticker_symbol": "AAPL"}Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The latest price of Apple (AAPL) is $234.06 as of September 12, 2025, 15:59.

As you can see, the task execution is decomposed into explicit reasoning "Thought", action, and observation steps.

By making each step observable and traceable, it enables fine-grained debugging and helps identify precisely where issues originate during execution.


## With Memory

In this code snippet, we add (working) memory (Context) to the agent so we can keep track of the conversations and ask several related questions.

In [ ]:
from llama_index.core.agent.workflow import AgentStream

from llama_index.core.workflow import Context

agent_prices = ReActAgent(
    llm=OpenAI(model="gpt-4o", api_key = OPENAI_API_KEY), tools=tools, timeout=120, verbose=False
)

ctx = Context(agent_prices)

handler_1 = agent_prices.run("What is the last price of Apple?", ctx = ctx)
async for ev in handler_1.stream_events():
    if isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)

response = await handler_1


handler_2 = agent_prices.run("Which date was it?", ctx=ctx)
async for ev in handler_2.stream_events():
    if isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)

response = await handler_2

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_latest_price
Action Input: {"ticker_symbol": "AAPL"}Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The latest price of Apple (AAPL) is $234.06 as of September 12, 2025.

Thought: The user is asking for the date of the latest price information I provided.
Answer: The date of the latest price information for Apple (AAPL) is September 12, 2025.

Memory management is handled through the Context object provided by LlamaIndex. This context is linked to the agent and initialized when the agent is executed.

In the first query, we asked to retrieve the last price of Apple. The ReAct agent runs the reasoning (Thought) and acting (invoking the get_latest_price tool) traces, and provide this answer:

> Answer: The latest price of Apple (AAPL) is $234.06 as of September 12, 2025

We then ask a second question to retrieve the date. As shown above, the agent only performs the thought step and does not call the tool, since it already obtained the answer from the first interaction which was kept in memory.

Finally, we get the final result:

> The date of the latest price information for Apple (AAPL) is September 12, 2025.



